In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
silver_df = spark.table("predictive_cataglog.bronze_layer.bronze_table")


In [0]:
silver_df.limit(5).display()

engine_id,cycle,op1,op2,op3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21,ingestion_ts
1,1,-7.0E-4,-4.0E-4,100.0,518.67,641.82,1589.7,1400.6,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.419,2026-01-13T06:56:46.546Z
1,2,0.0019,-3.0E-4,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.0,23.4236,2026-01-13T06:56:46.546Z
1,3,-0.0043,3.0E-4,100.0,518.67,642.35,1587.99,1404.2,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,2026-01-13T06:56:46.546Z
1,4,7.0E-4,0.0,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,2026-01-13T06:56:46.546Z
1,5,-0.0019,-2.0E-4,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.0,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.8,8.4294,0.03,393,2388,100.0,38.9,23.4044,2026-01-13T06:56:46.546Z


In [0]:
silver_df.count()

20631

In [0]:
engine_window = Window.partitionBy("engine_id").orderBy("cycle")
rolling_window = engine_window.rowsBetween(-5, 0)

In [0]:
for i in range(1, 22):
    silver_df = silver_df \
        .withColumn(f"s{i}_roll_mean", avg(f"s{i}").over(rolling_window)) \
        .withColumn(f"s{i}_delta", silver_df[f"s{i}"] - lag(f"s{i}").over(engine_window))

In [0]:
silver_df = silver_df.fillna(0)


In [0]:
silver_df1=silver_df.select(
    "engine_id", "cycle", "op1", "op2", "op3",
    *[c for c in silver_df.columns if "roll_mean" in c or "delta" in c]
)

In [0]:
silver_df1.count()

20631

In [0]:
silver_df1.limit(5).display()
  

engine_id,cycle,op1,op2,op3,s1_roll_mean,s1_delta,s2_roll_mean,s2_delta,s3_roll_mean,s3_delta,s4_roll_mean,s4_delta,s5_roll_mean,s5_delta,s6_roll_mean,s6_delta,s7_roll_mean,s7_delta,s8_roll_mean,s8_delta,s9_roll_mean,s9_delta,s10_roll_mean,s10_delta,s11_roll_mean,s11_delta,s12_roll_mean,s12_delta,s13_roll_mean,s13_delta,s14_roll_mean,s14_delta,s15_roll_mean,s15_delta,s16_roll_mean,s16_delta,s17_roll_mean,s17_delta,s18_roll_mean,s18_delta,s19_roll_mean,s19_delta,s20_roll_mean,s20_delta,s21_roll_mean,s21_delta
1,1,-7.0E-4,-4.0E-4,100.0,518.67,0.0,641.82,0.0,1589.7,0.0,1400.6,0.0,14.62,0.0,21.61,0.0,554.36,0.0,2388.06,0.0,9046.19,0.0,1.3,0.0,47.47,0.0,521.66,0.0,2388.02,0.0,8138.62,0.0,8.4195,0.0,0.03,0.0,392.0,0,2388.0,0,100.0,0.0,39.06,0.0,23.419,0.0
1,2,0.0019,-3.0E-4,100.0,518.67,0.0,641.985,0.32999999999992724,1590.76,2.119999999999891,1401.87,2.540000000000191,14.62,0.0,21.61,0.0,554.0550000000001,-0.6100000000000136,2388.05,-0.01999999999998181,9045.130000000001,-2.1200000000008004,1.3,0.0,47.480000000000004,0.020000000000003126,521.97,0.6200000000000045,2388.045,0.0500000000001819,8135.055,-7.130000000000109,8.425650000000001,0.012300000000001532,0.03,0.0,392.0,0,2388.0,0,100.0,0.0,39.03,-0.060000000000002274,23.421300000000002,0.0045999999999999375
1,3,-0.0043,3.0E-4,100.0,518.67,0.0,642.1066666666667,0.20000000000004547,1589.8366666666668,-3.8299999999999272,1402.6466666666665,1.0599999999999454,14.62,0.0,21.61,0.0,554.1233333333333,0.5099999999999909,2388.06,0.03999999999996362,9047.733333333335,8.8700000000008,1.3,0.0,47.410000000000004,-0.21999999999999886,522.12,0.13999999999998636,2388.0400000000004,-0.03999999999996362,8134.446666666667,1.7399999999997817,8.423033333333334,-0.014000000000001123,0.03,0.0,391.3333333333333,-2,2388.0,0,100.0,0.0,39.00333333333334,-0.04999999999999716,23.3956,-0.07939999999999969
1,4,7.0E-4,0.0,100.0,518.67,0.0,642.1675,0.0,1588.075,-5.2000000000000455,1402.4524999999999,-2.3300000000001546,14.62,0.0,21.61,0.0,554.205,0.19000000000005457,2388.0725,0.03000000000020009,9048.170000000002,-3.460000000000946,1.3,0.0,47.34,-0.14000000000000057,522.3050000000001,0.44000000000005457,2388.05,0.04999999999972715,8134.2925,0.6000000000003638,8.409325,-0.049599999999999866,0.03,0.0,391.5,2,2388.0,0,100.0,0.0,38.972500000000004,-0.07000000000000028,23.390175,0.029699999999998283
1,5,-0.0019,-2.0E-4,100.0,518.67,0.0,642.208,0.01999999999998181,1587.03,0.05999999999994543,1403.206,4.350000000000136,14.62,0.0,21.61,0.0,554.164,-0.4500000000000455,2388.07,-0.0500000000001819,9049.566000000003,5.670000000000073,1.3,0.0,47.328,0.14999999999999858,522.282,-0.6699999999999591,2388.0480000000002,-0.03999999999996362,8134.194,-0.02999999999974534,8.413340000000002,0.06119999999999948,0.03,0.0,391.8,1,2388.0,0,100.0,0.0,38.958000000000006,0.01999999999999602,23.39302,0.03049999999999997


In [0]:
%sql
create database if not exists predictive_cataglog.silver_layer

In [0]:
silver_df1.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("predictive_cataglog.silver_layer.silver_table")

In [0]:
specific_row = silver_df1.filter(silver_df1['engine_id'] == 100).first()
print(specific_row)

Row(engine_id=100, cycle=1, op1=-0.0033, op2=0.0003, op3=100.0, s1_roll_mean=518.67, s1_delta=0.0, s2_roll_mean=642.25, s2_delta=0.0, s3_roll_mean=1596.57, s3_delta=0.0, s4_roll_mean=1404.52, s4_delta=0.0, s5_roll_mean=14.62, s5_delta=0.0, s6_roll_mean=21.61, s6_delta=0.0, s7_roll_mean=553.99, s7_delta=0.0, s8_roll_mean=2388.07, s8_delta=0.0, s9_roll_mean=9055.97, s9_delta=0.0, s10_roll_mean=1.3, s10_delta=0.0, s11_roll_mean=47.36, s11_delta=0.0, s12_roll_mean=522.13, s12_delta=0.0, s13_roll_mean=2388.14, s13_delta=0.0, s14_roll_mean=8146.76, s14_delta=0.0, s15_roll_mean=8.4177, s15_delta=0.0, s16_roll_mean=0.03, s16_delta=0.0, s17_roll_mean=393.0, s17_delta=0, s18_roll_mean=2388.0, s18_delta=0, s19_roll_mean=100.0, s19_delta=0.0, s20_roll_mean=38.72, s20_delta=0.0, s21_roll_mean=23.3899, s21_delta=0.0)


In [0]:
%sql
select * from predictive_cataglog.silver_layer.silver_table limit 5

engine_id,cycle,op1,op2,op3,s1_roll_mean,s1_delta,s2_roll_mean,s2_delta,s3_roll_mean,s3_delta,s4_roll_mean,s4_delta,s5_roll_mean,s5_delta,s6_roll_mean,s6_delta,s7_roll_mean,s7_delta,s8_roll_mean,s8_delta,s9_roll_mean,s9_delta,s10_roll_mean,s10_delta,s11_roll_mean,s11_delta,s12_roll_mean,s12_delta,s13_roll_mean,s13_delta,s14_roll_mean,s14_delta,s15_roll_mean,s15_delta,s16_roll_mean,s16_delta,s17_roll_mean,s17_delta,s18_roll_mean,s18_delta,s19_roll_mean,s19_delta,s20_roll_mean,s20_delta,s21_roll_mean,s21_delta
1,1,-7.0E-4,-4.0E-4,100.0,518.67,0.0,641.82,0.0,1589.7,0.0,1400.6,0.0,14.62,0.0,21.61,0.0,554.36,0.0,2388.06,0.0,9046.19,0.0,1.3,0.0,47.47,0.0,521.66,0.0,2388.02,0.0,8138.62,0.0,8.4195,0.0,0.03,0.0,392.0,0,2388.0,0,100.0,0.0,39.06,0.0,23.419,0.0
1,2,0.0019,-3.0E-4,100.0,518.67,0.0,641.985,0.32999999999992724,1590.76,2.119999999999891,1401.87,2.540000000000191,14.62,0.0,21.61,0.0,554.0550000000001,-0.6100000000000136,2388.05,-0.01999999999998181,9045.130000000001,-2.1200000000008004,1.3,0.0,47.480000000000004,0.020000000000003126,521.97,0.6200000000000045,2388.045,0.0500000000001819,8135.055,-7.130000000000109,8.425650000000001,0.012300000000001532,0.03,0.0,392.0,0,2388.0,0,100.0,0.0,39.03,-0.060000000000002274,23.421300000000002,0.0045999999999999375
1,3,-0.0043,3.0E-4,100.0,518.67,0.0,642.1066666666667,0.20000000000004547,1589.8366666666668,-3.8299999999999272,1402.6466666666665,1.0599999999999454,14.62,0.0,21.61,0.0,554.1233333333333,0.5099999999999909,2388.06,0.03999999999996362,9047.733333333335,8.8700000000008,1.3,0.0,47.410000000000004,-0.21999999999999886,522.12,0.13999999999998636,2388.0400000000004,-0.03999999999996362,8134.446666666667,1.7399999999997817,8.423033333333334,-0.014000000000001123,0.03,0.0,391.3333333333333,-2,2388.0,0,100.0,0.0,39.00333333333334,-0.04999999999999716,23.3956,-0.07939999999999969
1,4,7.0E-4,0.0,100.0,518.67,0.0,642.1675,0.0,1588.075,-5.2000000000000455,1402.4524999999999,-2.3300000000001546,14.62,0.0,21.61,0.0,554.205,0.19000000000005457,2388.0725,0.03000000000020009,9048.170000000002,-3.460000000000946,1.3,0.0,47.34,-0.14000000000000057,522.3050000000001,0.44000000000005457,2388.05,0.04999999999972715,8134.2925,0.6000000000003638,8.409325,-0.049599999999999866,0.03,0.0,391.5,2,2388.0,0,100.0,0.0,38.972500000000004,-0.07000000000000028,23.390175,0.029699999999998283
1,5,-0.0019,-2.0E-4,100.0,518.67,0.0,642.208,0.01999999999998181,1587.03,0.05999999999994543,1403.206,4.350000000000136,14.62,0.0,21.61,0.0,554.164,-0.4500000000000455,2388.07,-0.0500000000001819,9049.566000000003,5.670000000000073,1.3,0.0,47.328,0.14999999999999858,522.282,-0.6699999999999591,2388.0480000000002,-0.03999999999996362,8134.194,-0.02999999999974534,8.413340000000002,0.06119999999999948,0.03,0.0,391.8,1,2388.0,0,100.0,0.0,38.958000000000006,0.01999999999999602,23.39302,0.03049999999999997
